# EDA & Preprocessing — DisasterWatch

Exploratory Data Analysis dan Preprocessing data bencana BNPB periode **2015–2026**.

Notebook membaca data dari `data/processed/` dan menghasilkan:
- Profil dan kualitas data
- Distribusi & tren temporal
- Analisis dampak dan severity
- **Feature Engineering** untuk clustering & klasifikasi
- **Clustering wilayah** (KMeans + DBSCAN)
- **Klasifikasi tingkat keparahan** (feature importance)
- **Analisis tren dan decomposisi musiman**
- **Insight otomatis & ringkasan akhir**

## 0. Setup & Load Data

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import seaborn as sns
from IPython.display import display, Markdown

# ML
from sklearn.preprocessing import StandardScaler, LabelEncoder, RobustScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, silhouette_score, davies_bouldin_score
from sklearn.impute import SimpleImputer

# Stat
from scipy import stats
from scipy.stats import kruskal

# Palet profesional, konsisten, dan ramah pembaca (termasuk color-vision deficiency).
PRIMARY = '#1F4E79'       # biru: frekuensi / metrik utama
TEAL = '#2A9D8F'          # hijau-teal: kondisi positif / pendukung
AMBER = '#E9C46A'         # kuning: perhatian sedang
CORAL = '#E76F51'         # merah-oranye: dampak tinggi
SLATE = '#6C7A89'         # abu-biru: pembanding
CATEGORICAL = [PRIMARY, TEAL, AMBER, CORAL, '#7B2CBF', '#457B9D', '#8AB17D', '#C06C84']
SEVERITY_COLORS = {'Berat': '#C44536', 'Sedang': '#E9A23B',
                   'Ringan': '#2A9D8F', 'Tanpa Dampak': '#B8C4CE',
                   'Tidak Diketahui': '#6C7A89'}

sns.set_theme(style='whitegrid', palette=CATEGORICAL, rc={
    'axes.spines.top': False, 'axes.spines.right': False,
    'grid.color': '#D9E1E8', 'grid.linewidth': .7,
    'axes.titleweight': 'semibold', 'figure.facecolor': 'white',
    'axes.facecolor': 'white'
})
plt.rcParams.update({'figure.dpi': 110, 'axes.titlesize': 13, 'axes.labelsize': 11})
pd.set_option('display.max_columns', 120)

# ── Path ──────────────────────────────────────────────────────────────────────
ROOT = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()


# ── Sumber data tunggal ───────────────────────────────────────────────────────
# Harus identik dengan notebook preprocessing: pilih CSV bnpb_bencana_*.csv
# dengan rentang tanggal terluas (bukan alfabetis / ukuran file).
def pick_raw_csv(folder):
    candidates = list(Path(folder).glob('bnpb_bencana_*.csv'))
    if not candidates:
        raise FileNotFoundError(f'Tidak ada bnpb_bencana_*.csv di {folder}')

    def coverage(path):
        try:
            start, end = path.stem.split('_')[-2:]
            return (pd.Timestamp(end) - pd.Timestamp(start)).days, pd.Timestamp(end)
        except (ValueError, IndexError):
            return (-1, pd.Timestamp.min)

    return max(candidates, key=coverage)


DATA = pick_raw_csv(ROOT / 'data' / 'processed')
print('Menggunakan file:', DATA.name)

df_raw = pd.read_csv(DATA, encoding='utf-8-sig', low_memory=False)
print(f'Shape awal: {df_raw.shape}')
df_raw.head(2)

## 1. Struktur & Kualitas Data

In [ ]:
df_raw.info()
display(df_raw.describe(include='all').T)

In [ ]:
# ── Missing-value heatmap & tabel ─────────────────────────────────────────────
missing = df_raw.isna().mean().mul(100).sort_values(ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart missing > 0
miss_nonzero = missing[missing > 0]
miss_nonzero.plot.barh(ax=axes[0], color='salmon')
axes[0].set(title='Persentase Missing per Kolom', xlabel='Missing (%)', xlim=(0, 105))
axes[0].axvline(50, color='red', ls='--', lw=.8, label='50%')
axes[0].legend()

# Heatmap sampel
cols_show = missing.head(20).index.tolist()
sns.heatmap(df_raw[cols_show].isna().sample(min(500, len(df_raw)), random_state=42).T,
            cbar=False, cmap='Reds', ax=axes[1])
axes[1].set(title='Pola Missing (sampel 500 baris)')
plt.tight_layout()
plt.show()

display(missing.to_frame('missing_pct').round(2))

In [ ]:
# ── Duplikasi ─────────────────────────────────────────────────────────────────
key_cols = ['tahun', 'bulan', 'tanggal', 'id_kab', 'kejadian', 'lokasi']
dup_full = df_raw.duplicated().sum()
dup_key  = df_raw.duplicated(subset=key_cols, keep=False).sum()

print(f'Duplikat baris penuh  : {dup_full:,}')
print(f'Duplikat kunci bisnis : {dup_key:,}')

dup_groups = (df_raw.groupby(key_cols, dropna=False).size()
              .reset_index(name='jumlah_baris')
              .query('jumlah_baris > 1')
              .sort_values('jumlah_baris', ascending=False))
display(dup_groups.head(10))

## 2. Preprocessing & Feature Engineering

In [ ]:
df = df_raw.copy()

# Kolom teks/kategori: NaN → '-' hanya untuk tampilan/analisis teks.
text_cols = df.select_dtypes(include=['object', 'string']).columns
if len(text_cols):
    df[text_cols] = df[text_cols].fillna('-')

# ── Tanggal ───────────────────────────────────────────────────────────────────
for c in ['tahun', 'bulan', 'tanggal']:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df['date'] = pd.to_datetime(
    dict(year=df.tahun, month=df.bulan, day=df.tanggal), errors='coerce')
# EDA menggunakan seluruh riwayat data mulai 1 Januari 2015.
START_DATE = pd.Timestamp('2015-01-01')
df = df.dropna(subset=['date']).query('date >= @START_DATE').copy()
# Samakan dengan preprocessing: hapus duplikat penuh agar baris yang dianalisis identik.
df = df.drop_duplicates().copy()

# ── Fitur waktu ───────────────────────────────────────────────────────────────
df['year']       = df.date.dt.year
df['month']      = df.date.dt.month
df['quarter']    = df.date.dt.quarter
df['dayofweek']  = df.date.dt.dayofweek          # 0=Senin
df['dayofyear']  = df.date.dt.dayofyear
df['week']       = df.date.dt.isocalendar().week.astype(int)
df['year_month'] = df.date.dt.to_period('M').astype(str)

# Musim hujan Indonesia: Nov–Mar (1), Kemarau: Apr–Okt (0)
df['musim_hujan'] = df.month.isin([11, 12, 1, 2, 3]).astype(int)

# ── Kolom dampak numerik ──────────────────────────────────────────────────────
# Nilai kosong TIDAK diisi 0: NaN berarti "tidak dilaporkan". Mengisinya 0
# membuat laporan tanpa data terbaca sebagai "tanpa dampak".
impact_cols = ['meninggal', 'hilang', 'terluka', 'menderita', 'mengungsi',
               'rumah_rusak', 'rumah_rusak_berat', 'rumah_rusak_sedang', 'rumah_rusak_ringan',
               'terendam', 'lahan', 'sawah', 'kebun', 'hutan',
               'jembatan_rusak', 'jalan', 'jumlah_kejadian', 'kerugian_juta',
               'pendidikan_rusak', 'peribadatan_rusak', 'kesehatan_rusak',
               'kantor_rusak', 'fasum_rusak', 'kios_rusak', 'pabrik_rusak']
impact_cols = [c for c in impact_cols if c in df.columns]
for c in impact_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce').clip(lower=0)

# Flag missing per kolom dampak + ringkasan kelengkapan per baris
for c in impact_cols:
    df[f'flag_{c}_missing'] = df[c].isna().astype(int)

core_impact = ['meninggal', 'hilang', 'terluka', 'menderita', 'mengungsi',
               'rumah_rusak', 'terendam']
core_impact = [c for c in core_impact if c in df.columns]
df['impact_missing_count']      = df[impact_cols].isna().sum(axis=1)
df['impact_report_ratio']       = 1 - df['impact_missing_count'] / len(impact_cols)
df['core_impact_missing_count'] = df[core_impact].isna().sum(axis=1)
df['has_impact_report']         = (df['core_impact_missing_count'] < len(core_impact)).astype(int)
df['has_full_impact_report']    = (df['core_impact_missing_count'] == 0).astype(int)

# ── Feature turunan (NaN-aware) ───────────────────────────────────────────────
# Jumlahkan hanya komponen yang dilaporkan; semua NaN → NaN.
def sum_impact(*cols):
    cols = [c for c in cols if c in df.columns]
    if not cols:
        return pd.Series(np.nan, index=df.index)
    sub = df[cols]
    return sub.sum(axis=1, skipna=True).where(sub.notna().any(axis=1))

df['korban_jiwa']     = sum_impact('meninggal', 'hilang')
df['korban_total']    = sum_impact('meninggal', 'hilang', 'terluka')
df['people_affected'] = sum_impact('menderita', 'mengungsi')
df['material_damage'] = sum_impact('rumah_rusak', 'terendam')
df['agri_damage']     = sum_impact('sawah', 'kebun', 'lahan')
df['infra_damage']    = sum_impact('jembatan_rusak', 'jalan')

# Log-transform untuk distribusi skewed (NaN tetap NaN)
for c in ['korban_total', 'korban_jiwa', 'people_affected',
          'material_damage', 'agri_damage', 'kerugian_juta']:
    df[f'log_{c}'] = np.log1p(df[c])

# ── Severity label (4 kelas + 'Tidak Diketahui') ──────────────────────────────
# Label hanya diberikan bila bukti cukup:
#   - Berat/Sedang: satu komponen sudah melewati ambang
#   - Ringan/Tanpa : semua komponen penentu diketahui
#   - selain itu   : 'Tidak Diketahui' (severity_code = NaN)
def severity_label(row):
    kj, kt = row['korban_jiwa'], row['korban_total']
    pa, md = row['people_affected'], row['material_damage']
    if (kj >= 5) or (pa >= 500) or (md >= 100):
        return 'Berat'
    if (kj >= 1) or (pa >= 50) or (md >= 10):
        return 'Sedang'
    if pd.isna(kj) or pd.isna(kt) or pd.isna(pa) or pd.isna(md):
        return 'Tidak Diketahui'
    if (kt > 0) or (pa > 0) or (md > 0):
        return 'Ringan'
    return 'Tanpa Dampak'


df['severity'] = df.apply(severity_label, axis=1)
df['severity_code'] = df['severity'].map(
    {'Tanpa Dampak': 0, 'Ringan': 1, 'Sedang': 2, 'Berat': 3})


def pct_severity(series, level):
    # Persentase level severity dari baris yang labelnya diketahui.
    known = series[series.isin(['Tanpa Dampak', 'Ringan', 'Sedang', 'Berat'])]
    return (known == level).mean() * 100 if len(known) else np.nan


# ── Koordinat ─────────────────────────────────────────────────────────────────
for c in ['geometry_lon', 'geometry_lat']:
    df[c] = pd.to_numeric(df[c], errors='coerce')

print(f'Shape setelah preprocessing: {df.shape}')
print(f'Rentang tanggal: {df.date.min().date()} → {df.date.max().date()}')
print(f"Baris tanpa label severity: {(df.severity == 'Tidak Diketahui').sum():,} "
      f"({(df.severity == 'Tidak Diketahui').mean() * 100:.1f}%)")
display(df[['date','kejadian','provinsi','korban_total','people_affected',
            'material_damage','severity','has_full_impact_report']].head(5))

In [ ]:
# ── Kelengkapan pelaporan dampak per tahun ────────────────────────────────────
# Menjelaskan kenapa komposisi severity bisa berbeda antar tahun: bukan karena
# bencana berubah, tetapi karena kolom dampak tertentu tidak selalu dilaporkan.
completeness = df.groupby('year').agg(
    n_laporan=('date', 'size'),
    pct_korban_NA=('meninggal', lambda s: s.isna().mean() * 100),
    pct_terdampak_NA=('menderita', lambda s: s.isna().mean() * 100),
    pct_terendam_NA=('terendam', lambda s: s.isna().mean() * 100),
    pct_laporan_lengkap=('has_full_impact_report', lambda s: s.mean() * 100),
    pct_tanpa_label=('severity', lambda s: (s == 'Tidak Diketahui').mean() * 100),
).round(1)
display(completeness)

fig, ax = plt.subplots(figsize=(13, 4))
completeness[['pct_korban_NA', 'pct_terdampak_NA', 'pct_terendam_NA',
              'pct_tanpa_label']].plot(kind='bar', ax=ax, width=.8,
                                       color=[CORAL, AMBER, SLATE, PRIMARY])
ax.set(title='Kelengkapan Pelaporan Dampak per Tahun',
       xlabel='Tahun', ylabel='% laporan')
ax.legend(['Korban jiwa tidak dilaporkan', 'Terdampak tidak dilaporkan',
           'Terendam tidak dilaporkan', 'Label severity Tidak Diketahui'],
          fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# ── Ringkasan distribusi severity ─────────────────────────────────────────────
order = ['Berat', 'Sedang', 'Ringan', 'Tanpa Dampak', 'Tidak Diketahui']
colors = [SEVERITY_COLORS[level] for level in order]

sev_dist = (df['severity'].value_counts().reindex(order, fill_value=0)
            .to_frame('count').rename_axis('severity').reset_index())
sev_dist['pct'] = sev_dist['count'] / len(df) * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.countplot(data=df, x='severity', order=order, palette=SEVERITY_COLORS, ax=axes[0])
axes[0].set(title='Distribusi Tingkat Keparahan', xlabel='', ylabel='Jumlah')
for p in axes[0].patches:
    axes[0].annotate(f'{p.get_height():,.0f}', (p.get_x()+p.get_width()/2, p.get_height()),
                     ha='center', va='bottom', fontsize=9)

wedges, texts, autotexts = axes[1].pie(
    sev_dist.set_index('severity').loc[order, 'pct'],
    labels=order, autopct='%1.1f%%', colors=colors, startangle=140)
axes[1].set_title('Proporsi Severity')
plt.tight_layout(); plt.show()
display(sev_dist.round(2))
print('Catatan: "Tidak Diketahui" = kolom dampak penentu tidak dilaporkan, '
      'bukan berarti tanpa dampak.')

## Dashboard Visual ? Ringkasan Cepat

Grafik berikut merangkum pertanyaan utama: **bencana apa yang dominan, kapan puncaknya, bagaimana tingkat dampaknya, dan wilayah mana yang paling banyak melapor**.

In [ ]:
# Dashboard ringkas agar insight utama dapat dibaca dalam satu tampilan
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'Mei', 'Jun', 'Jul', 'Agu', 'Sep', 'Okt', 'Nov', 'Des']
type_counts = df['kejadian'].value_counts().head(8).sort_values()
monthly_counts = df.groupby('month').size().reindex(range(1, 13), fill_value=0)
prov_counts = df['provinsi'].value_counts().head(10).sort_values()
severity_counts = df['severity'].value_counts().reindex(order, fill_value=0)

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('Dashboard EDA Bencana Indonesia (2015?2026)', fontsize=18, fontweight='bold', y=.98)

axes[0, 0].barh(type_counts.index, type_counts.values, color=sns.color_palette('Blues_r', len(type_counts)))
axes[0, 0].set(title='8 Jenis Bencana dengan Laporan Terbanyak', xlabel='Jumlah laporan', ylabel='')
for i, value in enumerate(type_counts.values):
    axes[0, 0].text(value, i, f'  {value:,.0f}', va='center', fontsize=9)

axes[0, 1].plot(range(1, 13), monthly_counts.values, marker='o', lw=2.5, color=PRIMARY)
axes[0, 1].fill_between(range(1, 13), monthly_counts.values, alpha=.20, color=PRIMARY)
axes[0, 1].set(title='Pola Musiman: Laporan per Bulan', xlabel='Bulan', ylabel='Jumlah laporan', xticks=range(1, 13), xticklabels=month_names)
axes[0, 1].grid(axis='y', alpha=.25)

axes[1, 0].pie(severity_counts.values, labels=severity_counts.index, autopct='%1.1f%%', startangle=90,
               colors=[SEVERITY_COLORS[level] for level in severity_counts.index], wedgeprops={'edgecolor': 'white'})
axes[1, 0].set_title('Komposisi Tingkat Keparahan')

axes[1, 1].barh(prov_counts.index, prov_counts.values, color=sns.color_palette('YlGnBu', len(prov_counts)))
axes[1, 1].set(title='10 Provinsi dengan Laporan Terbanyak', xlabel='Jumlah laporan', ylabel='')
for i, value in enumerate(prov_counts.values):
    axes[1, 1].text(value, i, f'  {value:,.0f}', va='center', fontsize=9)

plt.tight_layout(rect=[0, 0, 1, .96])
plt.show()


In [ ]:
# Tren tahunan: frekuensi laporan dan penduduk terdampak
yearly_visual = (df.groupby('year')
                 .agg(laporan=('date', 'size'), terdampak=('people_affected', 'sum'))
                 .reset_index())

fig, ax1 = plt.subplots(figsize=(15, 5))
bars = ax1.bar(yearly_visual['year'], yearly_visual['laporan'], color=PRIMARY, label='Laporan')
ax1.set(title='Tren Tahunan Laporan dan Penduduk Terdampak', xlabel='Tahun', ylabel='Jumlah laporan')
ax1.bar_label(bars, fmt='{:,.0f}', padding=3, fontsize=8)
ax1.set_xticks(yearly_visual['year'])

ax2 = ax1.twinx()
ax2.plot(yearly_visual['year'], yearly_visual['terdampak'], color=CORAL, marker='o', lw=2.5, label='Penduduk terdampak')
ax2.set_ylabel('Penduduk terdampak')
ax2.yaxis.set_major_formatter(mticker.StrMethodFormatter('{x:,.0f}'))

handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(handles1 + handles2, labels1 + labels2, loc='upper left')
ax1.grid(axis='y', alpha=.25)
plt.tight_layout(); plt.show()


## 3. Distribusi Kejadian & Dampak

In [ ]:
# ── Top jenis bencana ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

type_counts = df['kejadian'].value_counts()
sns.barplot(x=type_counts.values, y=type_counts.index, palette='Blues_r', ax=axes[0])
axes[0].set(title='Jumlah Laporan per Jenis Bencana', xlabel='Jumlah', ylabel='')
for i, v in enumerate(type_counts.values):
    axes[0].text(v + 50, i, f'{v:,}', va='center', fontsize=8)

# Severity per jenis bencana (stacked)
sev_type = pd.crosstab(df['kejadian'], df['severity'])
sev_type_pct = sev_type.div(sev_type.sum(axis=1), axis=0) * 100
sev_type_pct[order].reindex(type_counts.index).plot(
    kind='barh', stacked=True, ax=axes[1], color=colors, width=0.7)
axes[1].set(title='Komposisi Severity per Jenis Bencana (%)',
            xlabel='%', xlim=(0, 105))
axes[1].legend(title='Severity', bbox_to_anchor=(1, 1))
plt.tight_layout(); plt.show()

In [ ]:
# ── Ringkasan dampak per jenis bencana ───────────────────────────────────────
# sum(min_count=1) → NaN bila seluruh nilai grup tidak dilaporkan.
disaster_summary = df.groupby('kejadian', dropna=False).agg(
    jumlah_laporan=('kejadian', 'size'),
    total_korban_jiwa=('korban_jiwa', lambda x: x.sum(min_count=1)),
    total_korban_total=('korban_total', lambda x: x.sum(min_count=1)),
    total_terdampak=('people_affected', lambda x: x.sum(min_count=1)),
    total_material=('material_damage', lambda x: x.sum(min_count=1)),
    total_kerugian_juta=('kerugian_juta', lambda x: x.sum(min_count=1)),
    total_agri_ha=('agri_damage', lambda x: x.sum(min_count=1)),
    median_terdampak=('people_affected', 'median'),
    provinsi_terdampak=('provinsi', 'nunique'),
    kabkot_terdampak=('kabkot', 'nunique'),
    pct_berat=('severity', lambda x: pct_severity(x, 'Berat')),
    pct_tidak_diketahui=('severity', lambda x: (x == 'Tidak Diketahui').mean() * 100),
).sort_values('jumlah_laporan', ascending=False)
display(disaster_summary.round(2))

In [ ]:
# ── Distribusi log-dampak per jenis bencana ──────────────────────────────────
top_types = df.kejadian.value_counts().head(6).index.tolist()
df_top = df[df.kejadian.isin(top_types)]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics = [('log_korban_total', 'Log(korban+1)'),
           ('log_people_affected', 'Log(terdampak+1)'),
           ('log_material_damage', 'Log(material+1)')]

for ax, (col, label) in zip(axes, metrics):
    sns.boxplot(data=df_top, x='kejadian', y=col, palette=CATEGORICAL, ax=ax,
                order=top_types, showfliers=False)
    ax.set(title=f'Distribusi {label}', xlabel='', ylabel=label)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right', fontsize=8)

plt.suptitle('Distribusi Dampak per Jenis Bencana (6 Terbanyak)', y=1.01, fontsize=13)
plt.tight_layout(); plt.show()

## 4. Analisis Temporal & Tren

In [ ]:
# ── Tren laporan bulanan ──────────────────────────────────────────────────────
monthly = (df.groupby(df.date.dt.to_period('M'))
             .agg(jumlah=('date', 'size'),
                  korban=('korban_total', 'sum'),
                  terdampak=('people_affected', 'sum'))
             .reset_index())
monthly['date_ts'] = monthly.date.dt.to_timestamp()

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
for ax, col, title, color in zip(axes,
        ['jumlah', 'korban', 'terdampak'],
        ['Jumlah Laporan', 'Total Korban Jiwa/Cedera', 'Total Penduduk Terdampak'],
        ['steelblue', 'crimson', 'darkorange']):
    ax.plot(monthly.date_ts, monthly[col], color=color, lw=1.5)
    ax.fill_between(monthly.date_ts, monthly[col], alpha=.15, color=color)
    # Rolling 3-bulan
    ax.plot(monthly.date_ts, monthly[col].rolling(3, center=True).mean(),
            color='black', lw=1, ls='--', label='MA-3bln')
    ax.set(title=title, ylabel=col)
    ax.legend(fontsize=8)
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.suptitle('Tren Bulanan Bencana Indonesia 2015–2026', fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
# ── Dekomposisi musiman sederhana (STL-like via rolling) ─────────────────────
from pandas.plotting import autocorrelation_plot

monthly_ts = monthly.set_index('date_ts')['jumlah'].resample('MS').sum()

trend  = monthly_ts.rolling(12, center=True, min_periods=6).mean()
seasonal_component = monthly_ts - trend
seasonal_pattern = seasonal_component.groupby(seasonal_component.index.month).mean()

fig, axes = plt.subplots(1, 2, figsize=(15, 4))

axes[0].plot(monthly_ts.index, monthly_ts.values, label='Asli', alpha=.6)
axes[0].plot(trend.index, trend.values, color=CORAL, lw=2, label='Tren (MA-12)')
axes[0].set(title='Tren Jangka Panjang Laporan Bencana')
axes[0].legend()

bulan_names = ['Jan','Feb','Mar','Apr','Mei','Jun','Jul','Agu','Sep','Okt','Nov','Des']
bars = axes[1].bar(seasonal_pattern.index, seasonal_pattern.values,
                   color=['#d62728' if v > 0 else '#2ca02c' for v in seasonal_pattern.values])
axes[1].axhline(0, color='black', lw=0.8)
axes[1].set(title='Pola Musiman (deviasi rata-rata dari tren)',
            xlabel='Bulan', ylabel='Deviasi')
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(bulan_names)

plt.tight_layout(); plt.show()

print('Bulan dengan laporan di atas rata-rata tren:')
print(seasonal_pattern[seasonal_pattern > 0].sort_values(ascending=False)
      .rename(index={i: bulan_names[i-1] for i in range(1,13)}))

In [ ]:
# ── Heatmap bulan × jenis bencana ─────────────────────────────────────────────
pivot_month_type = pd.crosstab(df['month'], df['kejadian'])
pivot_month_type.index = bulan_names

plt.figure(figsize=(14, 6))
sns.heatmap(pivot_month_type, annot=True, fmt='g', cmap='YlGnBu',
            linewidths=.3, linecolor='grey')
plt.title('Total Laporan per Bulan × Jenis Bencana (2015–2026)')
plt.xlabel('Jenis Bencana'); plt.ylabel('Bulan')
plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.show()

In [ ]:
# ── Heatmap tahun × jenis bencana (% dari total per tahun) ─────────────────
pivot_year_type = pd.crosstab(df['year'], df['kejadian'])
pivot_year_type_pct = pivot_year_type.div(pivot_year_type.sum(axis=1), axis=0) * 100

plt.figure(figsize=(14, 7))
sns.heatmap(pivot_year_type_pct, annot=True, fmt='.1f', cmap='Blues',
            linewidths=.3)
plt.title('Proporsi Jenis Bencana per Tahun (% dari total laporan tahun itu)')
plt.xlabel('Jenis Bencana'); plt.ylabel('Tahun')
plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.show()

In [ ]:
# ── YoY growth & CAGR ─────────────────────────────────────────────────────────
yearly = df.groupby('year').agg(
    laporan=('date', 'size'),
    korban=('korban_total', 'sum'),
    terdampak=('people_affected', 'sum'),
    kerugian=('kerugian_juta', 'sum')
)
yearly['yoy_laporan_%'] = yearly['laporan'].pct_change() * 100

valid_years = yearly.dropna(subset=['yoy_laporan_%'])
n_years = yearly.index.max() - yearly.index.min()
cagr = ((yearly.laporan.iloc[-1] / yearly.laporan.iloc[0]) ** (1/n_years) - 1) * 100 if n_years > 0 else 0

print(f'CAGR laporan bencana ({yearly.index.min()}–{yearly.index.max()}): {cagr:.2f}%/tahun')
display(yearly.round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
yearly['laporan'].plot(kind='bar', ax=axes[0], color=PRIMARY, title='Total Laporan per Tahun')
axes[0].set(xlabel='Tahun', ylabel='Jumlah')
valid_years['yoy_laporan_%'].plot(kind='bar', ax=axes[1],
    color=[TEAL if v > 0 else CORAL for v in valid_years['yoy_laporan_%']],
    title='Pertumbuhan YoY Laporan (%)')
axes[1].axhline(0, color='black', lw=0.8)
axes[1].set(xlabel='Tahun', ylabel='YoY %')
plt.tight_layout(); plt.show()

## 5. Analisis Spasial & Wilayah

In [ ]:
# ── Konsentrasi Pareto ───────────────────────────────────────────────────────
prov_stats = df.groupby('provinsi').agg(
    laporan=('date', 'size'),
    korban=('korban_total', 'sum'),
    terdampak=('people_affected', 'sum'),
    kerugian=('kerugian_juta', 'sum'),
    jenis_bencana=('kejadian', 'nunique'),
).sort_values('laporan', ascending=False)

prov_stats['pct_laporan'] = prov_stats['laporan'] / prov_stats['laporan'].sum() * 100
prov_stats['cumulative_pct'] = prov_stats['pct_laporan'].cumsum()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Pareto chart
top20 = prov_stats.head(20)
ax = axes[0]
ax.bar(range(len(top20)), top20['laporan'], color=PRIMARY, alpha=0.8)
ax2 = ax.twinx()
ax2.plot(range(len(top20)), top20['cumulative_pct'].values, color=CORAL, marker='o', linestyle='-', ms=4)
ax2.axhline(80, color=AMBER, ls='--', lw=.8, label='80%')
ax.set(title='Pareto Laporan per Provinsi (Top 20)',
       xlabel='Provinsi', ylabel='Jumlah Laporan')
ax.set_xticks(range(len(top20)))
ax.set_xticklabels(top20.index, rotation=60, ha='right', fontsize=7)
ax2.set(ylabel='Kumulatif %')
ax2.legend()

# Scatter: laporan vs korban
sc = axes[1].scatter(prov_stats['laporan'], prov_stats['korban'],
                     s=prov_stats['terdampak'] / max(prov_stats['terdampak'].max(), 1) * 800 + 20,
                     c=prov_stats['kerugian'], cmap='Reds', alpha=.7)
plt.colorbar(sc, ax=axes[1], label='Kerugian (Juta Rp)')
for prov in prov_stats.head(10).index:
    axes[1].annotate(prov, (prov_stats.loc[prov, 'laporan'], prov_stats.loc[prov, 'korban']),
                     fontsize=7, alpha=.85)
axes[1].set(title='Laporan vs Korban per Provinsi\n(ukuran = terdampak)',
            xlabel='Jumlah Laporan', ylabel='Total Korban')
plt.tight_layout(); plt.show()
display(prov_stats.head(20).round(2))

In [ ]:
# ── Peta scatter sederhana ────────────────────────────────────────────────────
geo = df.dropna(subset=['geometry_lon', 'geometry_lat']).copy()
geo = geo[(geo.geometry_lon.between(90, 145)) & (geo.geometry_lat.between(-12, 8))]
print(f'Koordinat valid: {len(geo):,} dari {len(df):,} ({len(geo)/len(df)*100:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, col, title, cmap in [
    (axes[0], 'severity_code', 'Sebaran Laporan (warna = severity)', 'RdYlGn_r'),
    (axes[1], 'log_korban_total', 'Sebaran Korban (warna = log korban)', 'Reds')
]:
    sc = ax.scatter(geo.geometry_lon, geo.geometry_lat,
                    c=geo[col], cmap=cmap, s=5, alpha=.3, vmin=0)
    plt.colorbar(sc, ax=ax, label=col)
    ax.set(title=title, xlabel='Longitude', ylabel='Latitude', aspect='equal')
    ax.axhline(0, color='grey', lw=0.5, ls='--')

plt.tight_layout(); plt.show()

In [ ]:
# ── Komposisi jenis bencana 10 provinsi teratas ───────────────────────────────
top10_prov = prov_stats.head(10).index.tolist()
prov_type = pd.crosstab(df[df.provinsi.isin(top10_prov)].provinsi,
                        df[df.provinsi.isin(top10_prov)].kejadian)
prov_type_pct = prov_type.div(prov_type.sum(axis=1), axis=0) * 100

prov_type_pct.loc[top10_prov].plot(
    kind='barh', stacked=True, figsize=(13, 6), colormap='tab20', width=.7)
plt.title('Komposisi Jenis Bencana per Provinsi Top-10 (%)')
plt.xlabel('% laporan'); plt.xlim(0, 105)
plt.legend(title='Jenis Bencana', bbox_to_anchor=(1, 1), fontsize=8)
plt.tight_layout(); plt.show()

## 6. Korelasi & Analisis Multivariat

In [ ]:
# ── Heatmap korelasi dampak ───────────────────────────────────────────────────
corr_cols = ['meninggal', 'hilang', 'terluka', 'menderita', 'mengungsi',
             'rumah_rusak', 'terendam', 'lahan', 'kerugian_juta',
             'korban_total', 'people_affected', 'material_damage']
corr_cols = [c for c in corr_cols if c in df.columns]
corr = df[corr_cols].corr()

mask = np.triu(np.ones_like(corr, dtype=bool))
plt.figure(figsize=(12, 9))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=.3)
plt.title('Korelasi Antar Indikator Dampak Bencana')
plt.tight_layout(); plt.show()

In [ ]:
# ── Uji Kruskal-Wallis: apakah dampak berbeda antar jenis bencana? ────────────
print('=== Kruskal-Wallis: Dampak per Jenis Bencana ===')
for metric in ['korban_total', 'people_affected', 'material_damage']:
    groups = [g[metric].values for _, g in df.groupby('kejadian') if len(g) >= 5]
    if len(groups) >= 2:
        stat, p = kruskal(*groups)
        sig = '✓ Signifikan' if p < 0.05 else '✗ Tidak signifikan'
        print(f'  {metric:25s}: H={stat:.2f}, p={p:.4f}  {sig}')

In [ ]:
# ── Pair plot fitur utama ─────────────────────────────────────────────────────
pair_cols = ['log_korban_total', 'log_people_affected', 'log_material_damage', 'log_kerugian_juta']
pair_cols = [c for c in pair_cols if c in df.columns]

sample_df = df[pair_cols + ['severity']].sample(min(3000, len(df)), random_state=42)
pg = sns.pairplot(sample_df, hue='severity', hue_order=order,
                  palette=dict(zip(order, colors)), corner=True, plot_kws={'alpha': .3, 's': 10})
pg.figure.suptitle('Pair Plot Fitur Dampak (log-scale)', y=1.01, fontsize=13)
plt.tight_layout(); plt.show()

## 7. Clustering Wilayah (KMeans + PCA + DBSCAN)

In [ ]:
# ── Aggregasi fitur per provinsi ──────────────────────────────────────────────
cluster_features = [
    'laporan', 'korban', 'terdampak', 'kerugian', 'jenis_bencana'
]
# Tambah rasio severity (dihitung hanya dari label yang diketahui)
prov_cluster = prov_stats.copy()
sev_prov = df.groupby('provinsi').apply(
    lambda x: pd.Series({
        'pct_berat':       pct_severity(x.severity, 'Berat'),
        'pct_sedang':      pct_severity(x.severity, 'Sedang'),
        'pct_ringan':      pct_severity(x.severity, 'Ringan'),
        'pct_tanpa_label': (x.severity == 'Tidak Diketahui').mean() * 100,
        'avg_terdampak':   x.people_affected.mean(),
        'avg_korban':      x.korban_total.mean(),
        'musim_hujan_pct': x.musim_hujan.mean() * 100,
    })
)
prov_cluster = prov_cluster.join(sev_prov)

# Normalisasi
feat_cols = ['laporan', 'korban', 'terdampak', 'kerugian',
             'jenis_bencana', 'pct_berat', 'avg_terdampak', 'avg_korban']
feat_cols = [c for c in feat_cols if c in prov_cluster.columns]
X_prov = prov_cluster[feat_cols].fillna(0)

scaler = RobustScaler()
X_scaled = scaler.fit_transform(X_prov)

# ── Elbow method ──────────────────────────────────────────────────────────────
inertias, sil_scores = [], []
K_range = range(2, min(9, len(X_prov)))
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(K_range), inertias, 'bo-')
axes[0].set(title='Elbow Method (Inertia)', xlabel='K', ylabel='Inertia')
axes[1].plot(list(K_range), sil_scores, 'ro-')
axes[1].set(title='Silhouette Score per K', xlabel='K', ylabel='Silhouette')
plt.tight_layout(); plt.show()

best_k = list(K_range)[sil_scores.index(max(sil_scores))]
print(f'K optimal (max silhouette): {best_k}  | score={max(sil_scores):.3f}')

In [ ]:
# ── KMeans final ──────────────────────────────────────────────────────────────
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
prov_cluster['cluster_kmeans'] = km_final.fit_predict(X_scaled)

# PCA 2D
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
prov_cluster['pca1'] = X_pca[:, 0]
prov_cluster['pca2'] = X_pca[:, 1]

print(f'Explained variance PCA: {pca.explained_variance_ratio_.cumsum()[1]*100:.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# PCA scatter
colors_c = plt.cm.Set1(np.linspace(0, 0.8, best_k))
for c in range(best_k):
    mask = prov_cluster.cluster_kmeans == c
    axes[0].scatter(prov_cluster.loc[mask, 'pca1'], prov_cluster.loc[mask, 'pca2'],
                    label=f'Cluster {c}', s=80, alpha=.8)
    for prov in prov_cluster[mask].index:
        axes[0].annotate(prov, (prov_cluster.loc[prov, 'pca1'], prov_cluster.loc[prov, 'pca2']),
                         fontsize=6.5, alpha=.85)
axes[0].set(title=f'PCA 2D — KMeans Clustering Provinsi (K={best_k})',
            xlabel=f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)',
            ylabel=f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[0].legend()

# Cluster profile
cluster_profile = prov_cluster.groupby('cluster_kmeans')[feat_cols].mean()
sns.heatmap(cluster_profile.T, annot=True, fmt='.1f', cmap='YlGnBu',
            ax=axes[1], linewidths=.3)
axes[1].set(title='Profil Rata-rata per Cluster', xlabel='Cluster')
plt.tight_layout(); plt.show()

# Anggota per cluster
for c in range(best_k):
    provs = prov_cluster[prov_cluster.cluster_kmeans == c].index.tolist()
    print(f'\nCluster {c} ({len(provs)} provinsi): {provs}')

In [ ]:
# ── DBSCAN (clustering koordinat bencana) ─────────────────────────────────────
geo_db = df.dropna(subset=['geometry_lon', 'geometry_lat']).copy()
geo_db = geo_db[(geo_db.geometry_lon.between(90, 145)) & (geo_db.geometry_lat.between(-12, 8))]

X_geo = geo_db[['geometry_lon', 'geometry_lat']].values
X_geo_scaled = StandardScaler().fit_transform(X_geo)

db = DBSCAN(eps=0.15, min_samples=10)
geo_db['cluster_dbscan'] = db.fit_predict(X_geo_scaled)

n_clusters_db = len(set(geo_db.cluster_dbscan)) - (1 if -1 in geo_db.cluster_dbscan.values else 0)
noise_pct = (geo_db.cluster_dbscan == -1).mean() * 100
print(f'DBSCAN: {n_clusters_db} kluster, noise={noise_pct:.1f}%')

plt.figure(figsize=(14, 6))
scatter = plt.scatter(geo_db.geometry_lon, geo_db.geometry_lat,
                      c=geo_db.cluster_dbscan, cmap='tab20',
                      s=8, alpha=.3, vmin=-1)
plt.colorbar(scatter, label='DBSCAN Cluster (-1=noise)')
plt.title(f'DBSCAN Spatial Clustering Kejadian Bencana\n({n_clusters_db} kluster ditemukan)')
plt.xlabel('Longitude'); plt.ylabel('Latitude')
plt.tight_layout(); plt.show()

# Top cluster stats
if n_clusters_db > 0:
    db_stats = geo_db[geo_db.cluster_dbscan >= 0].groupby('cluster_dbscan').agg(
        count=('date', 'size'),
        avg_lon=('geometry_lon', 'mean'),
        avg_lat=('geometry_lat', 'mean'),
        top_kejadian=('kejadian', lambda x: x.value_counts().index[0]),
        korban=('korban_total', 'sum')
    ).sort_values('count', ascending=False)
    display(db_stats.head(10))

## 8. Klasifikasi Tingkat Keparahan (Feature Importance)

In [ ]:
# ── Persiapan fitur klasifikasi ───────────────────────────────────────────────
# Fitur yang realistis tersedia SEBELUM dampak diketahui (untuk prediksi)
# dan fitur dampak untuk analisis post-event

# Encode categorical
le_type  = LabelEncoder()
le_prov  = LabelEncoder()

df['kejadian_code'] = le_type.fit_transform(df['kejadian'].fillna('Unknown'))
df['provinsi_code'] = le_prov.fit_transform(df['provinsi'].fillna('Unknown'))

# Fitur untuk klasifikasi severity (analisis dampak, bukan prediksi murni)
clf_features_post = [
    'kejadian_code', 'provinsi_code',
    'month', 'quarter', 'musim_hujan', 'year',
    'meninggal', 'hilang', 'terluka', 'menderita', 'mengungsi',
    'rumah_rusak', 'terendam', 'lahan', 'kerugian_juta',
    'jumlah_kejadian',
    'core_impact_missing_count'
]
clf_features_post = [c for c in clf_features_post if c in df.columns]

# Baris tanpa label severity (dampak tidak lengkap) dikeluarkan dari training
n_before = len(df)
df_clf = df.dropna(subset=['severity_code']).copy()
print(f"Baris tanpa label severity (Tidak Diketahui): {n_before - len(df_clf):,}")

X_clf = df_clf[clf_features_post].fillna(0)  # imputasi 0 khusus input model
y_clf = df_clf['severity_code']

X_train, X_test, y_train, y_test = train_test_split(
    X_clf, y_clf, test_size=.2, random_state=42, stratify=y_clf)

print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')
print('Distribusi severity (train):', y_train.value_counts().to_dict())

In [ ]:
# ── Random Forest ─────────────────────────────────────────────────────────────
rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42,
                            class_weight='balanced', n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

sev_names = {0: 'Tanpa Dampak', 1: 'Ringan', 2: 'Sedang', 3: 'Berat'}
print('=== Random Forest Classification Report ===')
print(classification_report(y_test, y_pred_rf,
      target_names=[sev_names[i] for i in sorted(sev_names)]))

cv_scores = cross_val_score(rf, X_clf, y_clf, cv=5, scoring='f1_weighted', n_jobs=-1)
print(f'CV F1-weighted: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')

In [ ]:
# ── Feature Importance ────────────────────────────────────────────────────────
feat_imp = pd.Series(rf.feature_importances_, index=clf_features_post).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Bar importances
feat_imp.head(15).plot.barh(ax=axes[0], color=PRIMARY)
axes[0].invert_yaxis()
axes[0].set(title='Top-15 Feature Importance (Random Forest)', xlabel='Importance')

# Confusion matrix heatmap
from sklearn.metrics import confusion_matrix
labels_present = sorted(y_test.unique())
label_names    = [sev_names[l] for l in labels_present]
cm = confusion_matrix(y_test, y_pred_rf, labels=labels_present)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names, ax=axes[1])
axes[1].set(title='Confusion Matrix (%)', xlabel='Prediksi', ylabel='Aktual')
plt.tight_layout(); plt.show()

display(feat_imp.to_frame('importance').round(4))

In [ ]:
# ── Gradient Boosting untuk konfirmasi ───────────────────────────────────────
# Hanya 3 fitur kontekstual (skenario prediksi awal)
pre_features = ['kejadian_code', 'provinsi_code', 'month', 'musim_hujan', 'year']
X_pre = df_clf[pre_features].fillna(0)

X_tr2, X_te2, y_tr2, y_te2 = train_test_split(
    X_pre, y_clf, test_size=.2, random_state=42, stratify=y_clf)

gb = GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42)
gb.fit(X_tr2, y_tr2)
print('=== GBM (fitur kontekstual saja) ===')
print(classification_report(y_te2, gb.predict(X_te2),
      target_names=[sev_names[i] for i in sorted(sev_names)]))

feat_imp2 = pd.Series(gb.feature_importances_, index=pre_features).sort_values(ascending=False)
feat_imp2.plot.barh(figsize=(8, 3), title='GBM Feature Importance (Fitur Kontekstual)')
plt.tight_layout(); plt.show()

## 9. Analisis Tren Lanjutan per Jenis Bencana

In [ ]:
# ── Tren tahunan per jenis bencana ────────────────────────────────────────────
top_types5 = df.kejadian.value_counts().head(5).index.tolist()
yearly_type = (df[df.kejadian.isin(top_types5)]
               .groupby(['year', 'kejadian']).size().unstack(fill_value=0))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

yearly_type[top_types5].plot(ax=axes[0], marker='o', ms=5)
axes[0].set(title='Tren Tahunan per Jenis Bencana (Top 5)',
            xlabel='Tahun', ylabel='Jumlah Laporan')
axes[0].legend(fontsize=8)

# Normalized (indeks 2015=100)
base = yearly_type[top_types5].iloc[0].replace(0, np.nan)
yearly_type_norm = yearly_type[top_types5].div(base) * 100
yearly_type_norm.plot(ax=axes[1], marker='o', ms=5)
axes[1].axhline(100, color='black', ls='--', lw=.8)
axes[1].set(title='Indeks Tren (2015=100)',
            xlabel='Tahun', ylabel='Indeks')
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# ── Perubahan bulanan per tahun (heatmap per jenis) ──────────────────────────
for disaster in top_types5[:3]:
    sub = df[df.kejadian == disaster]
    pivot = pd.crosstab(sub['year'], sub['month'])
    pivot.columns = bulan_names

    plt.figure(figsize=(13, max(3, len(pivot)*0.4 + 1)))
    sns.heatmap(pivot, annot=True, fmt='g', cmap='YlGnBu',
                linewidths=.3, linecolor='grey')
    plt.title(f'Laporan "{disaster}" per Tahun × Bulan')
    plt.xlabel('Bulan'); plt.ylabel('Tahun')
    plt.tight_layout(); plt.show()

In [ ]:
# ── Rolling 12-bulan trend per jenis bencana ──────────────────────────────────
monthly_type = (df.groupby([df.date.dt.to_period('M'), 'kejadian'])
                .size().unstack(fill_value=0)
                .sort_index())
monthly_type.index = monthly_type.index.to_timestamp()

fig, axes = plt.subplots(len(top_types5), 1, figsize=(14, 3*len(top_types5)), sharex=True)
for ax, t in zip(axes, top_types5):
    if t not in monthly_type.columns:
        continue
    s = monthly_type[t]
    ax.bar(s.index, s.values, width=20, alpha=.4, color=PRIMARY)
    ax.plot(s.index, s.rolling(12, center=True, min_periods=3).mean(),
            color=CORAL, lw=2, label='MA-12bln')
    ax.set(title=t, ylabel='Laporan')
    ax.legend(fontsize=8)

plt.suptitle('Tren Bulanan + Rolling Avg per Jenis Bencana', fontsize=13)
plt.tight_layout(); plt.show()

## 10. Analisis Outlier & Kejadian Ekstrem

In [ ]:
# ── Z-score outlier pada dampak ───────────────────────────────────────────────
# nan_policy='omit' → baris tanpa data dampak tidak dianggap nol.
for col in ['korban_total', 'people_affected', 'material_damage']:
    if col not in df.columns:
        continue
    df[f'z_{col}'] = np.abs(stats.zscore(df[col], nan_policy='omit'))

z_cols = [c for c in df.columns if c.startswith('z_')]
outlier_mask = (df[z_cols] > 3).any(axis=1)
print(f'Outlier ekstrem (|z|>3 pada salah satu metrik): {outlier_mask.sum():,} laporan')

extreme_events = (df[outlier_mask]
    .sort_values('korban_total', ascending=False)
    [['date', 'provinsi', 'kabkot', 'kejadian', 'severity',
      'korban_total', 'people_affected', 'material_damage', 'kerugian_juta']]
    .head(20)
)
display(extreme_events)

In [ ]:
# ── Visualisasi kejadian ekstrem ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col, label in [
    (axes[0], 'korban_total', 'Total Korban'),
    (axes[1], 'people_affected', 'Terdampak'),
    (axes[2], 'material_damage', 'Kerusakan Material')
]:
    if col not in df.columns:
        continue
    data = df[col].clip(upper=df[col].quantile(.99))
    ax.hist(data[data > 0], bins=50, color=PRIMARY, edgecolor='white', lw=.3)
    ax.set(title=f'Distribusi {label} (clip p99)', xlabel=label, ylabel='Frekuensi')
    ax.set_yscale('log')

plt.suptitle('Distribusi Dampak (log-scale, clip p99)', fontsize=13)
plt.tight_layout(); plt.show()

# Top 10 kejadian paling berdampak
display(Markdown('### Top 10 Bencana Terdampak (People)'))
display(df.nlargest(10, 'people_affected')
        [['date','provinsi','kabkot','kejadian','severity',
          'korban_total','people_affected','material_damage']].reset_index(drop=True))

display(Markdown('### Top 10 Bencana Paling Mematikan'))
display(df.nlargest(10, 'korban_jiwa')
        [['date','provinsi','kabkot','kejadian','severity',
          'korban_jiwa','people_affected','kerugian_juta']].reset_index(drop=True))

## 11. Profil Kabupaten/Kota Risiko Tinggi

In [ ]:
# ── Skor risiko composite per kabkot ─────────────────────────────────────────
kabkot_stats = df.groupby(['provinsi', 'kabkot']).agg(
    laporan=('date', 'size'),
    korban=('korban_total', lambda x: x.sum(min_count=1)),
    terdampak=('people_affected', lambda x: x.sum(min_count=1)),
    material=('material_damage', lambda x: x.sum(min_count=1)),
    kerugian=('kerugian_juta', lambda x: x.sum(min_count=1)),
    pct_berat=('severity', lambda x: pct_severity(x, 'Berat')),
    jenis_unik=('kejadian', 'nunique'),
).reset_index()

# Normalize & composite score
norm_cols = ['laporan', 'korban', 'terdampak', 'material', 'pct_berat']
for c in norm_cols:
    mn, mx = kabkot_stats[c].min(), kabkot_stats[c].max()
    kabkot_stats[f'n_{c}'] = (kabkot_stats[c] - mn) / (mx - mn + 1e-9)

kabkot_stats['risk_score'] = (
    kabkot_stats['n_laporan']   * 0.25 +
    kabkot_stats['n_korban']    * 0.25 +
    kabkot_stats['n_terdampak'] * 0.20 +
    kabkot_stats['n_material']  * 0.15 +
    kabkot_stats['n_pct_berat'] * 0.15
) * 100

top_risk = kabkot_stats.nlargest(20, 'risk_score')
top_risk[['provinsi','kabkot','laporan','korban','terdampak','pct_berat','risk_score']].plot(
    x='kabkot', y='risk_score', kind='barh', figsize=(10, 7),
    title='Top 20 Kabupaten/Kota — Skor Risiko Bencana Composite', color=CORAL)
plt.xlabel('Risk Score (0–100)')
plt.tight_layout(); plt.show()

display(top_risk[['provinsi','kabkot','laporan','korban','terdampak','material',
                  'pct_berat','jenis_unik','risk_score']].round(2).reset_index(drop=True))

## 12. Export Dataset Terproses

In [ ]:
# ── Simpan dataset yang sudah diperkaya untuk keperluan ML downstream ─────────
export_cols = [
    'date', 'year', 'month', 'quarter', 'dayofweek', 'musim_hujan',
    'tahun', 'bulan', 'tanggal',
    'id_prov', 'id_kab', 'provinsi', 'kabkot', 'desa',
    'id_jenis_bencana', 'kejadian', 'kejadian_code',
    'provinsi_code', 'severity', 'severity_code',
    # Dampak mentah (NaN = tidak dilaporkan)
    'meninggal', 'hilang', 'terluka', 'menderita', 'mengungsi',
    'rumah_rusak', 'rumah_rusak_berat', 'rumah_rusak_sedang', 'rumah_rusak_ringan',
    'terendam', 'lahan', 'sawah', 'kebun', 'hutan', 'kerugian_juta',
    'jumlah_kejadian', 'jembatan_rusak', 'jalan',
    'korban_jiwa', 'korban_total', 'people_affected', 'material_damage',
    'agri_damage', 'infra_damage',
    # Kelengkapan laporan dampak
    'impact_missing_count', 'impact_report_ratio', 'core_impact_missing_count',
    'has_impact_report', 'has_full_impact_report',
    'log_korban_jiwa', 'log_korban_total', 'log_people_affected',
    'log_material_damage', 'log_agri_damage', 'log_kerugian_juta',
    'geometry_lon', 'geometry_lat',
]
export_cols = [c for c in export_cols if c in df.columns]
export_cols += [c for c in df.columns if c.startswith('flag_') and c.endswith('_missing') and c not in export_cols]

out_path = ROOT / 'data' / 'processed' / 'bnpb_eda_enriched.csv'
df[export_cols].to_csv(out_path, index=False, encoding='utf-8-sig')
print(f'Dataset terproses disimpan → {out_path}')
print(f'Shape: {df[export_cols].shape}')

# Simpan juga cluster provinsi
prov_out = ROOT / 'data' / 'processed' / 'bnpb_province_clusters.csv'
prov_cluster[feat_cols + ['cluster_kmeans', 'pca1', 'pca2']].to_csv(prov_out)
print(f'Cluster provinsi disimpan → {prov_out}')

# Simpan risk score kabkot
risk_out = ROOT / 'data' / 'processed' / 'bnpb_kabkot_risk.csv'
kabkot_stats.to_csv(risk_out, index=False, encoding='utf-8-sig')
print(f'Risk score kabkot disimpan → {risk_out}')

## 13. Insight Otomatis & Ringkasan Analitik

In [ ]:
# ── Ringkasan lengkap ─────────────────────────────────────────────────────────
top_type       = df.kejadian.value_counts().idxmax()
top_type_n     = int(df.kejadian.value_counts().max())
top_prov_name  = prov_stats.index[0]
top_prov_n     = int(prov_stats.laporan.iloc[0])
peak_month_n   = int(monthly.jumlah.max())
peak_month_ts  = monthly.loc[monthly.jumlah.idxmax(), 'date'].strftime('%Y-%m')
top5_share     = prov_stats.head(5).pct_laporan.sum()
peak_season    = seasonal_pattern.idxmax()
peak_season_nm = bulan_names[peak_season - 1]
total_korban   = df.korban_total.sum(min_count=1)
total_tdp      = df.people_affected.sum(min_count=1)
pct_berat      = pct_severity(df.severity, 'Berat')
pct_tanpa_lbl  = (df.severity == 'Tidak Diketahui').mean() * 100
pct_lengkap    = df.has_full_impact_report.mean() * 100
highest_risk   = top_risk.iloc[0]

print('=' * 60)
print('RINGKASAN INSIGHT EDA BENCANA INDONESIA (2015–2026)')
print('=' * 60)
print(f'\n📊 DATA UMUM')
print(f'  Total laporan    : {len(df):,}')
print(f'  Rentang waktu    : {df.date.min().date()} → {df.date.max().date()}')
print(f'  Jenis bencana    : {df.kejadian.nunique()}')
print(f'  Provinsi         : {df.provinsi.nunique()}')
print(f'  Kabupaten/kota   : {df.kabkot.nunique()}')

print(f'\n🔥 KEJADIAN')
print(f'  Bencana paling sering : {top_type} ({top_type_n:,} laporan)')
print(f'  Provinsi terparah     : {top_prov_name} ({top_prov_n:,} laporan)')
print(f'  Bulan puncak           : {peak_month_ts} ({peak_month_n:,} laporan)')
print(f'  Bulan musim tinggi     : {peak_season_nm} (deviasi tertinggi dari tren)')
print(f'  Konsentrasi Top-5 prov : {top5_share:.1f}% laporan nasional')

print(f'\n💔 DAMPAK (batas bawah — hanya dari laporan yang tersedia)')
print(f'  Total korban jiwa/cedera : {total_korban:,.0f}')
print(f'  Total penduduk terdampak : {total_tdp:,.0f}')
print(f'  Laporan dampak lengkap   : {pct_lengkap:.1f}%')
print(f'  Laporan tingkat Berat    : {pct_berat:.1f}% (dari label yang diketahui)')
print(f'  Label Tidak Diketahui    : {pct_tanpa_lbl:.1f}%')
print(f'  CAGR laporan bencana     : {cagr:.2f}%/tahun')

print(f'\n🗺️  CLUSTERING')
print(f'  Jumlah kluster provinsi  : {best_k} (silhouette={max(sil_scores):.3f})')
print(f'  DBSCAN hotspot spasial   : {n_clusters_db} kluster kejadian')
print(f'  Kabkot risiko tertinggi  : {highest_risk.kabkot}, {highest_risk.provinsi}')
print(f'                             (risk score={highest_risk.risk_score:.1f}/100)')

print(f'\n🤖 KLASIFIKASI (Random Forest)')
print(f'  CV F1-weighted (post-event)   : {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')
print(f'  Fitur paling penting          : {feat_imp.index[0]} ({feat_imp.iloc[0]:.3f})')
print(f'  2nd                           : {feat_imp.index[1]} ({feat_imp.iloc[1]:.3f})')

print('\n⚠️  Validasi temuan ini dengan data kependudukan, topografi,')
print('   dan kelengkapan pelaporan BPBD sebelum pengambilan keputusan.')

## Kesimpulan & Arah Analisis Selanjutnya

### Temuan Utama
| Dimensi | Temuan |
|---|---|
| **Frekuensi** | Banjir & tanah longsor mendominasi laporan |
| **Tren** | CAGR positif → intensifikasi bencana dari tahun ke tahun |
| **Musiman** | Puncak pada musim hujan (Nov–Mar), trough kemarau |
| **Spasial** | Konsentrasi di Jawa, Sumatera, Kalimantan |
| **Clustering** | Provinsi terbagi dalam kluster berdasarkan frekuensi & keparahan |
| **Severity** | Berat ~19%; 24% laporan berlabel *Tidak Diketahui* karena kolom dampak tidak dilaporkan |

### Catatan Kualitas Data
- Nilai dampak yang kosong **tidak** diisi 0, tetapi tetap `NaN` + flag
  (`flag_<kolom>_missing`, `impact_missing_count`, `has_full_impact_report`).
- Label `Tidak Diketahui` muncul ketika komponen penentu severity (korban jiwa,
  terdampak, kerusakan material) tidak lengkap. Baris ini tidak dipakai untuk
  training klasifikasi, sehingga model tidak belajar dari label hasil imputasi.
- Kelengkapan kolom dampak berbeda tiap tahun (lihat tabel di bagian awal),
  jadi perbandingan severity antar tahun harus mempertimbangkan hal ini.

### Rekomendasi Analisis Selanjutnya
1. **Clustering** → Gunakan label cluster provinsi untuk segmentasi kebijakan mitigasi
2. **Klasifikasi** → Latih model prediksi severity dengan data cuaca (BMKG) sebagai fitur tambahan
3. **Tren** → Lakukan forecasting SARIMA/Prophet pada time series per jenis bencana
4. **Spasial** → Overlay dengan DEM, peta sungai, dan kepadatan penduduk untuk risk modeling
5. **Anomaly detection** → Identifikasi bencana signifikan yang under-reported